In [ ]:
import numpy as np
import copy
import plotly.graph_objects as go
from scipy.spatial import KDTree

# 1. IMPORTANT CLASSES

Below are the main classes used in this implementation.

The class **Bases_curves** constructs the B-spline basis functions in a discrete manner; that is, it evaluates them at the corresponding knot values and returns a scalar.

Next, the class **Nurbs_curves** allows the evaluation of a NURBS curve given the control points, weights, degree, and the corresponding knot vector.

The class **Construccion** builds the control points, weights, and knot vector required to represent a circle of radius $ r $.

The class **Nurbs_surface** evaluates a NURBS surface given the control point data (or control mesh), weights, degrees in both directions, and their corresponding knot vectors.

Finally, the class **Nurbs_surface3D_graph** enables the visualization of the surface based on the control point data, weights, knot vectors, and degrees.

It is important to emphasize that these classes do not construct any non-orientable surfaces; they are only intended for evaluating NURBS curves and surfaces, as well as for visualizing the resulting surface.

In [ ]:
class Bases_curves:
    def __init__(self, t=None):
        if t is None:
            t = [0, 1, 2, 3, 4]
        self.t = t

    def N_0(self, i, u):
        if i <= len(self.t) - 2:
            return (lambda u_val: 1 if ((u_val >= self.t[i]) & (u_val < self.t[i + 1])) else 0)(u)
        else:
            return 'El índice ha sobrepasado los límites'

    def N_p(self, i, u, p):
        if (p <= len(self.t) - 2) & (i <= len(self.t) - p - 2):
            if p == 0:
                return self.N_0(i, u)
            else:
                return ((lambda u_val: ((u_val - self.t[i]) / (self.t[i + p] - self.t[i]) * self.N_p(i, u, p - 1) 
                        if self.t[i + p] - self.t[i] != 0 else 0))(u) + 
                        (lambda u_val: ((self.t[i + p + 1] - u_val) / (self.t[i + p + 1] - self.t[i + 1]) * self.N_p(i + 1, u, p - 1) 
                        if self.t[i + p + 1] - self.t[i + 1] != 0 else 0))(u))
        else:
            return f'Ingrese valores de i=[0, {len(self.t) - p - 2}] y p=[0, {len(self.t) - 2}]'

In [ ]:
class Nurbs_curbs():
    def __init__(self):
        pass

    def curva(self, pts, pes, nod, p, u):
        suma_den, suma_num = 0, [0 for k in range(len(pts[0]))]
        for i in range(len(pes)):
            suma_den = suma_den + (lambda i_val: (pes[i_val]*Bases_curves(nod).N_p(i=i_val, u=u, p=p)))(i)

            suma_num = [a + b for a,b in zip(suma_num, (lambda i_val: ([
                pes[i_val]*Bases_curves(nod).N_p(i=i_val, u=u, p=p)*psdr for psdr in pts[i_val]]))(i)) ]
        
        return [psdr/suma_den for psdr in suma_num]

    def curva1(self, pts, pes, nod, p, u):
        if isinstance(u, (int, float)):  
            instancia = self.curva(pts, pes, nod, p, u)
            return instancia
        elif isinstance(u, (list, np.ndarray)):  
            instancias = [self.curva(pts, pes, nod, p, num) for num in u]
            return instancias
        else:
            raise ValueError("La entrada debe ser un número o un array de NumPy.")

In [ ]:
class Construccion():
    def __init__(self, n, r):
        self.n, self.r = n, r

    def polygon(self):
        θ = 2*np.pi/self.n
        α = np.pi*(self.n-2)/(2*self.n)
        R = self.r/np.sin(α)
        Tabla, Lista=[[[self.r*np.cos(θ*k),self.r*np.sin(θ*k)],[R*np.cos(θ*(2*k+1)/2),R*np.sin(θ*(2*k+1)/2)]] for k in range(self.n)], []
        for cont in range(len(Tabla)):
            Lista.extend([Tabla[cont][0], Tabla[cont][1]])
        Lista.append(Tabla[0][0])
        return Lista

    def peso(self):
        θ = 2*np.pi/self.n
        Tabla, Lista = [[1, np.cos(θ/2)] for k in range(self.n)], []
        for cont in range(len(Tabla)):
            Lista.extend(Tabla[cont])
        Lista.append(Tabla[0][0])
        return Lista

    def open1(self):
        Tabla, Lista = [[k,k] for k in range(self.n+1)], []
        for cont in range(len(Tabla)):
            Lista.extend(Tabla[cont])
        Lista = [0] + Lista + [self.n]
        return Lista

In [ ]:
class Nurbs_surface():
    def __init__(self, data, pes1, pes2, nod1, nod2, p1, p2):
        self.data, self.pes1, self.pes2, self.nod1, self.nod2, self.p1, self.p2 = data, pes1, pes2, nod1, nod2, p1, p2
        self.objeto1, self.objeto2 = Bases_curves(t=self.nod1), Bases_curves(t=self.nod2)

    def W_ij(self, i, j):
        w_ij = lambda i_val, j_val: (self.pes1[i_val]*self.pes2[j_val] if (i_val < len(self.pes1) and j_val < len(self.pes2))
                                       else 'índices sobrepasaron límites')
        return w_ij(i, j)

    def P_ij(self, i, j):
        p_ij = lambda i_val, j_val: (self.data[i][j])
        return np.array(p_ij(i,j), dtype=float)


    def superficie(self, u, v):
        # Número de puntos
        n1 = len(self.pes1)
        n2 = len(self.pes2)
    
        # Precalcular bases
        N_u = np.array([self.objeto1.N_p(i=i, u=u, p=self.p1) for i in range(n1)])
        N_v = np.array([self.objeto2.N_p(i=j, u=v, p=self.p2) for j in range(n2)])
    
        dim = len(self.data[0][0])
        suma_num = np.zeros(dim)
        suma_den = 0.0
    
        for i in range(n1):
            for j in range(n2):
                B = N_u[i] * N_v[j]
                W = self.W_ij(i, j)
                coef = B * W
    
                suma_den += coef
                suma_num += coef * np.array(self.P_ij(i, j))
    
        return suma_num / suma_den

    def procesar(self, u, v):
    
        if np.isscalar(u) and np.isscalar(v):
            return self.superficie(u, v)
    
        if isinstance(u, np.ndarray) and isinstance(v, np.ndarray):
    
            if u.shape != v.shape:
                raise ValueError("Ingrese vectores con mismas dimensiones")
    
            fila, columna = u.shape
            dim = len(self.data[0][0])
    
            resultado = np.zeros((fila, columna, dim))
    
            for i in range(fila):
                for j in range(columna):
                    resultado[i, j, :] = self.superficie(u[i, j], v[i, j])
    
            return [resultado[:, :, k] for k in range(dim)]
    
        raise ValueError("La entrada debe ser un número o un array de NumPy.")
        


In [ ]:
class Nurbs_superficie3D_graph:
    def __init__(self, data, pes1, pes2, nod1, nod2, p1, p2, ran_u, ran_v): # rango_u y rango_v depende de los valores de nod 1 y nod2
        self.data, self.pes1, self.pes2, self.nod1, self.nod2, self.p1, self.p2 = data, pes1, pes2, nod1, nod2, p1, p2
        self.ran_u, self.ran_v  = ran_u, ran_v
        self.u = np.linspace(self.ran_u[0], self.ran_u[1]-10**(-10), 50)
        self.v = np.linspace(self.ran_v[0], self.ran_v[1]-10**(-10), 50)
        self.U, self.V = np.meshgrid(self.u, self.v)
        self.objeto = Nurbs_surface(self.data, self.pes1, self.pes2, self.nod1, self.nod2, self.p1, self.p2)
        self.objeto1 = Nurbs_curbs()

    def grafica(self, enmaller=True, pts_ctrl=True, superficie=True, cur=True): 
        #datos para la superficie
        if superficie is True:    
            my_surface = self.objeto.procesar(u=self.U, v=self.V)
            X = my_surface[0]
            Y = my_surface[1]
            Z = my_surface[2]

        #datos para el enmallado
        def funci(lista):
            mi_lista = [[] for k in range(len(lista[0]))]
            for i in range(len(mi_lista)):
                mi_lista[i].append(np.array([k[i] for k in lista]))
            mi_lista = np.concatenate(np.array(mi_lista), axis=0)
            return mi_lista
            
        def funci1(matrix):
            columna = matrix.shape[1]
            lista_alm1 = [[] for k in range(columna)]
            for i in range(len(lista_alm1)):
                lista_alm1[i].append(matrix[: ,i])
            lista_alm1 = np.concatenate(np.array(lista_alm1), axis=0)
            return lista_alm1

        def grafica_maller(matrix):
            mi_lista1 = funci1(matrix)
            mi_lista2 = []
            for i in range(len(mi_lista1)):
                mi_lista2.append(funci(mi_lista1[i]))
            return mi_lista2
            
        #para el rango de la gráfica
        def min_max(data):
            n = len(data.shape)
            for i in range(n-1):
                data = np.concatenate(data, axis=0)
            return data

        r_m = min(min_max(self.data))-0.5
        r_M = max(min_max(self.data))+0.5

        #para puntos de control
        if pts_ctrl is True:     
            mi_puntos = funci(np.concatenate(self.data, axis=0))

        fig = go.Figure()
        if enmaller is True:
            lista_graf = grafica_maller(self.data)
            lista_graf1 = []
            for i in range(self.data.shape[0]):
                lista_graf1.append(funci(self.data[i, :]))
            for i in range(len(lista_graf)):
                fig.add_trace(go.Scatter3d(x=lista_graf[i][0], y=lista_graf[i][1], z=lista_graf[i][2], mode='lines', line=dict(color='blue'), showlegend=False))
            for i in range(len(lista_graf1)):
                fig.add_trace(go.Scatter3d(x=lista_graf1[i][0], y=lista_graf1[i][1], z=lista_graf1[i][2], mode='lines', line=dict(color='blue'), showlegend=False))
                
        if pts_ctrl is True:
            fig.add_trace(go.Scatter3d(x=mi_puntos[0], y=mi_puntos[1], z=mi_puntos[2], mode='markers', marker=dict(color='orange', size=5), showlegend=False))
            
        if superficie is True:
            fig.add_trace(go.Surface(z=Z, x=X, y=Y, colorscale='viridis', showscale=False))
            
            #lines = []
            #for i in range(X.shape[0]):
                #fig.add_trace(go.Scatter3d(x=X[i], y=Y[i], z=Z[i],
                                          #mode='lines',
                                          #line=dict(color='black', width=1),
                                          #showlegend=False))
            #for j in range(X.shape[1]):
                #fig.add_trace(go.Scatter3d(x=X[:,j], y=Y[:,j], z=Z[:,j],
                                          #mode='lines',
                                          #line=dict(color='black', width=1),
                                          #showlegend=False))

        if cur is True:
            f1 = funci(self.objeto1.curva1(pts=self.data[0], pes=self.pes2, nod=self.nod2, p=self.p2, u=self.v))
            f2 = funci(self.objeto1.curva1(pts=self.data[-1], pes=self.pes2, nod=self.nod2, p=self.p2, u=self.v))
            f3 = funci(self.objeto1.curva1(pts=self.data[:,0], pes=self.pes1, nod=self.nod1, p=self.p1, u=self.u))
            f4 = funci(self.objeto1.curva1(pts=self.data[:,-1], pes=self.pes1, nod=self.nod1, p=self.p1, u=self.u))
            fig.add_trace(go.Scatter3d(x=f1[0], y=f1[1], z=f1[2], mode='lines', line=dict(color='purple', width=8), showlegend=False))
            fig.add_trace(go.Scatter3d(x=f2[0], y=f2[1], z=f2[2], mode='lines', line=dict(color='purple', width=8), showlegend=False))
            fig.add_trace(go.Scatter3d(x=f3[0], y=f3[1], z=f3[2], mode='lines', line=dict(color='green', width=8), showlegend=False))
            fig.add_trace(go.Scatter3d(x=f4[0], y=f4[1], z=f4[2], mode='lines', line=dict(color='green', width=8), showlegend=False))
            


        
        fig.update_layout(
            scene=dict(
                xaxis=dict(showbackground=False, showgrid=False, showline=False, visible=False), #range=[r_m, r_M]
                yaxis=dict(showbackground=False, showgrid=False, showline=False, visible=False),
                zaxis=dict(showbackground=False, showgrid=False, showline=False, visible=False),
                aspectmode='data',  
                #aspectratio=dict(x=1, y=1, z=1)
            ),
            width=800,
            height=800
        )
        return fig

# 2. Class for Generating the Data of Non-Orientable Surfaces

This class generates the data, or control point mesh, required to construct non-orientable NURBS-type surfaces.

Below are the input parameters required to initialize the class:

**. pts1:** Control points of the rotation curve (unit circle), contained in the plane $ XY $.

**. pts3:** Control points of the twisting curve (unit semicircle), contained in the plane $ XZ $.

**. pts2:** Control points of the generatrix curve, contained in the plane $ XZ $. When defining this curve, it must be centered and, in general, it should be symmetric with respect to the $ Z $-axis (although this is not a necessary condition).

**. direction:** Represents the control points of the circle with radius $ a $, contained in the plane $ XY $.

$ Important: $

**-** The number of control points in **pts1**, **pts3**, and **direction** must be the same, and they must be defined over the same **knot** vector. Recall that these represent the circle and the semicircle.

**-** For **pts2**, the number of control points is independent of the other curves; therefore, its **knot** vector is also independent.

Once the class is initialized with the parameters described above, it automatically generates the data required to construct the non-orientable NURBS surface. This data can be accessed through the attribute **.data**.

**Construction of the data:** The control points (or control mesh) are obtained using the following formula and considerations:

Assume that the control points of **pts1** (unit circle) are of the form $ [R_{jx}, R_{jy}, 0] $, those of **pts3** (unit semicircle) are $ [T_{jx}, 0, T_{jz}] $, those of **direction** are $ a[R_{jx}, R_{jy}, 0] $, and finally those of **pts2** are $ [P_{ix}, 0, P_{iz}] $. Then, the control points of the surface mesh are given by:

$$ 
P_{ij} = a[R_{jx}, R_{jy},0] + [P_{ix}, 0, P_{iz}]
\begin{bmatrix}
T_{jx} & 0 & T_{jz} \\
0 & 1 & 0 \\
-T_{jz} & 0 & T_{jx}
\end{bmatrix}
\begin{bmatrix}
R_{jx} & R_{jy} & 0 \\
-R_{jy} & R_{jx} & 0 \\
0 & 0 & 1
\end{bmatrix} 
$$

$ Clarification: $ Since this class is intended to construct the data for non-orientable surfaces—and the topological definition of this type of surface consists of simultaneously rotating and twisting a curve—the same control points and corresponding **knot** vector will always be used for **pts1** and **pts3**.  

Regarding **direction**, the only variation is the radius of the circle. Therefore, in practical terms, it is only necessary to define the control points for **pts2**, along with its radius and its knot vector.

In [ ]:
class generation_data():
    def __init__(self, pts1, pts2, pts3, direction):
        self.pts1, self.pts2, self.pts3 = pts1, pts2, pts3
        self.direction = direction
        self._data()

    def _P_ij(self, i, j):
        p_ij = lambda i_val, j_val: ([self.direction[i_val][0]+self.pts2[j_val][0]*self.pts1[i_val][0]*self.pts3[i_val][0]-self.pts2[j_val][2]*self.pts1[i_val][0]*self.pts3[i_val][2],
                                      self.direction[i_val][1]+self.pts2[j_val][0]*self.pts1[i_val][1]*self.pts3[i_val][0]-self.pts2[j_val][2]*self.pts1[i_val][1]*self.pts3[i_val][2],
                                      self.pts2[j_val][0]*self.pts3[i_val][2]+self.pts2[j_val][2]*self.pts3[i_val][0]] 
                                     if (i_val < len(self.pts1) and j_val < len(self.pts2)) else 'índices sobrepasaron límites')
        return p_ij(i,j)

    def _data(self):
        data = []
        for i in range(len(self.pts1)):
            l_dat = []
            for j in range(len(self.pts2)):
                l_dat.append(self._P_ij(i, j))  
            data.append(l_dat)
        self.data = np.array(data, dtype=float)

# 3. Validation Method

This function colors the data points obtained from the NURBS surface according to their minimum distance with respect to the dense data points generated by evaluating the surface in its parametric form.

In [ ]:
def graficar_heatmap_error(data_original, data_densa):
    # 1. Calcula distancias mínimas
    arbol = KDTree(data_densa)
    distancias, _ = arbol.query(data_original)
    
    # 2. Se Crea la gráfica de dispersión 3D con Heatmap
    fig = go.Figure(data=[go.Scatter3d(
        x=data_original[:, 0],
        y=data_original[:, 1],
        z=data_original[:, 2],
        mode='markers',
        marker=dict(
            size=3,
            color=distancias,                # Color depende del error
            colorscale='Viridis',            
            colorbar=dict(title="Error (Distancia)"),
            showscale=True
        )
    )])

    # 3. Estética 
    fig.update_layout(
        template="plotly_white",
        title="Análisis de Error",
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    
    fig.show()
    
    # 4. Reporte estadístico 
    print(f"Error Medio: {np.mean(distancias):.6e}")
    print(f"Error Máximo: {np.max(distancias):.6e}")
    print(f"Desviación Estándar: {np.std(distancias):.6e}")

# 4. Validation Method Using the Normal Vector

This function receives the data $ X $, $ Y $, and $ Z $ returned by the class **Nurbs_surface(...).procesar(...)**. It plots the surface using this information, then selects a trajectory along one direction of the surface and displays its normal vectors.

This serves as a complementary verification to confirm that the data indeed corresponds to a non-orientable surface, if it is observed that the normal vectors reverse their direction after completing the trajectory.

In [ ]:
def normal_graph(X, Y, Z):
    # --- Cálculo de normales ---
    grad_x_u, grad_x_v = np.gradient(X)
    grad_y_u, grad_y_v = np.gradient(Y)
    grad_z_u, grad_z_v = np.gradient(Z)
    
    Nx = grad_y_u * grad_z_v - grad_z_u * grad_y_v
    Ny = grad_z_u * grad_x_v - grad_x_u * grad_z_v
    Nz = grad_x_u * grad_y_v - grad_y_u * grad_x_v
    
    # Normalización
    norm = np.sqrt(Nx**2 + Ny**2 + Nz**2)
    norm[norm == 0] = 1
    Nx, Ny, Nz = Nx/norm, Ny/norm, Nz/norm
    
    
    
    # --- CONFIGURACIÓN DE TAMAÑOS ---
    longitud_cuerpo = 0.4  
    tamaño_punta = 0.8     
    # --------------------------------
    
    # 1. Selección de datos (una fila de la malla)
    idx = X.shape[0] // 2
    # Se toma uno de cada 'n' puntos para que no se amontonen
    paso = 2 
    x_c, y_c, z_c = X[idx, ::paso], Y[idx, ::paso], Z[idx, ::paso]
    u_c, v_c, w_c = Nx[idx, ::paso], Ny[idx, ::paso], Nz[idx, ::paso]
    
    # 2. Crear las líneas (Cuerpos de las flechas)
    x_lines, y_lines, z_lines = [], [], []
    for i in range(len(x_c)):
        x_lines.extend([x_c[i], x_c[i] + u_c[i] * longitud_cuerpo, None])
        y_lines.extend([y_c[i], y_c[i] + v_c[i] * longitud_cuerpo, None])
        z_lines.extend([z_c[i], z_c[i] + w_c[i] * longitud_cuerpo, None])
    
    # 3. Gráfico
    fig = go.Figure()
    
    # Superficie de fondo
    fig.add_trace(go.Surface(x=X, y=Y, z=Z, opacity=0.3, showscale=False, colorscale='Greys'))
    
    # Curva trayectoria
    fig.add_trace(go.Scatter3d(
        x=X[idx], y=Y[idx], z=Z[idx],
        mode='lines',
        line=dict(color='blue', width=1),
        showlegend=False
    ))
    
    # Cuerpos de los vectores (Líneas)
    fig.add_trace(go.Scatter3d(
        x=x_lines, y=y_lines, z=z_lines,
        mode='lines',
        line=dict(color='black', width=6), 
        showlegend=False
    ))
    
    # Cabezas de los vectores (Conos)
    fig.add_trace(go.Cone(
        x=x_c + u_c * longitud_cuerpo, 
        y=y_c + v_c * longitud_cuerpo,
        z=z_c + w_c * longitud_cuerpo,
        u=u_c, v=v_c, w=w_c,
        sizemode="absolute",
        sizeref=tamaño_punta,
        anchor="tail",
        colorscale=[[0, 'red'], [1, 'red']], 
        showscale=False,
        name='Normal (Punta)'
    ))
    
    fig.update_layout(
        template="plotly_white",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False), 
            zaxis=dict(visible=False), 
            aspectmode='data',
            xaxis_title="X", yaxis_title="Y", zaxis_title="Z"
        ),
        margin=dict(l=0, r=0, b=0, t=30)
    )
    
    fig.show()

# 5. Examples:

Below, the control points of the rotation and twisting curves are constructed, together with their corresponding weights and knot vector. It should be noted that the degree is two, since we are dealing with a circle.

Likewise, these control points, weights, and knot vector will be used to generate the data for any non-orientable NURBS-type surface.

In [ ]:
# -------------------------------------------------------------------
# Definimos los puntos para la curva de rotación (pts1) y giro(pts3) 
# Este fragmento se repite para toda Superficie no orientable
# -------------------------------------------------------------------

# Puntos de pts1 en el plano XY
rot = Construccion(n=4, r=1).polygon()
rot = [psdr + [0] for psdr in rot]  

# Puntos de pts3 en el plano XZ
gir = Construccion(n=8, r=1).polygon()[ :len(rot) ]
gir = [psdr[:1] + [0] + psdr[1:] for psdr in gir] 

# Pesos de las curvas anteriores
w1 = Construccion(n=4, r=1).peso()

# Nodos de las curvas anteriores
x1 = Construccion(n=4, r=1).open1()

## 5.1. Möbius Strip

For this example, the Möbius strip is considered with the following parametric equations:

$$ 
S(u,v) = a
\begin{bmatrix}
\cos(u) + v \cos\left(\frac{u}{2}\right)\cos(u) \\
\sin(u) + v \cos\left(\frac{u}{2}\right)\sin(u) \\
v \sin\left(\frac{u}{2}\right)
\end{bmatrix},
\hspace{1mm} u \in [0,2\pi] \hspace{1mm} \wedge \hspace{1mm} v \in [-0.5,0.5]
$$

Below, the control points of the **direction** curve and the generatrix curve are specified. The user should pay particular attention to properly defining the control points of **direction** (i.e., the circle radius) and those of the generatrix curve, since the rotation and twisting curves remain the same for any non-orientable surface.

In this case, the generatrix curve is a straight line defined by two control points, and the **direction** curve coincides with the rotation curve, since the radius is $ a = 1 $.

In [ ]:
# ------------------------------------------------
# Definimos los puntos para la curva de direction
# ------------------------------------------------
direccion = rot


# -----------------------------------------------------
# Definimos los puntos para la curva generatriz (pts2) 
# -----------------------------------------------------

# Corresponde a una recta
gen = [[-0.5,0,0],[0.5,0,0]]

# Pesos de la curva generatriz
w2 = [1, 1]

# Nodos de la curva generatriz 
x2 = [0,0,1,1]

#### Data Generation:

The data (control mesh) for the Möbius strip is obtained by executing the following lines.

In [ ]:
ob = generation_data(pts1=rot, pts2=gen, pts3=gir, direction=direccion)
my_data_cinta = ob.data

#### Evaluation of the NURBS Surface

Once the data has been generated, the NURBS surface can be evaluated using the class **Nurbs_surface**. First, it is initialized, and then it is evaluated with the method **procesar**.

Note that $ u \in [0,4] $ and $ v \in [0,1] $, since the corresponding knot vectors in both directions are:
$ [0,0,0,1,1,2,2,3,3,4,4,4] \wedge [0,0,1,1] $.

It is advisable to subtract $ 10^{-10} $ to avoid division by zero. The following lines evaluate the NURBS surface with a resolution of $ 50 \times 50 $.

In [ ]:
# Rango de parámetros u y v
u = np.linspace(0, 4-10**(-10), 50)
v = np.linspace(0, 1-10**(-10), 50)
U, V = np.meshgrid(u,v)

# Evaluación de superficie NURBS
cinta = Nurbs_surface(my_data_cinta, w1, w2, x1, x2, 2, 1)
X, Y, Z  = cinta.procesar(U, V)

# Se cambia el formato a tamaño 2500x3 puntos
puntos1 = np.stack((X, Y, Z), axis=-1)
cinta_nurbs = puntos1.reshape(-1, 3, order='F')

##### Verification

Next, the normal vectors along a selected trajectory of the surface are visualized.  
If the normals reverse their direction, this confirms that the data indeed corresponds to a non-orientable surface.

In [ ]:
normal_graph(X, Y, Z)

#### Validation

First, let us evaluate the Möbius strip using its parametric equations with a resolution of $ 500 \times 500 $.

In [ ]:
# Rango de parámetros u y v
u = np.linspace(0, 2*np.pi, 500)
v = np.linspace(-0.5, 0.5, 500)
U, V = np.meshgrid(u,v)

# Evaluamos la superficie paramétrica
a = 1
X = a*(np.cos(U) + V*np.cos(U/2)*np.cos(U))
Y = a*(np.sin(U) + V*np.cos(U/2)*np.sin(U))
Z = a*V*np.sin(U/2)

# Se cambia el formato a tamaño 250000x3 puntos
puntos2 = np.stack((X, Y, Z), axis=-1)
cinta_parametrica = puntos2.reshape(-1, 3, order='F')

Next, we proceed to color the points of **cinta_nurbs** according to the minimum distance to all points in **cinta_parametrica**.  

This allows us to measure the error between the data generated using NURBS and that obtained through parametric evaluation, and thus verify that the data **cinta_nurbs** indeed corresponds to the Möbius strip.

In [ ]:
graficar_heatmap_error(cinta_nurbs, cinta_parametrica)

According to the plot, the error is very small, since most of the points in **cinta_nurbs** appear in dark colors, and according to the scale, these values correspond to distances that are nearly zero.

Furthermore, it can be observed that the maximum error is only 2.61%, with a standard deviation of 0.3%, ensuring the absence of significant outliers.

#### Comparison with the geomdl Library

Once this library has been imported, the data must be homogenized; that is, the points that originally have three coordinates will now have four, since the weight is being included.

This is done in order to use **geomdl** and construct the NURBS surface from the homogeneous control point data.

In [ ]:
from geomdl import NURBS

In [ ]:
data = []
pesos = []
m = len(w1)
n = len(w2)

for i in range(m):
    for j in range(n):
        data.append(cinta.P_ij(i,j))
        pesos.append(cinta.W_ij(i,j))

data_hom = [[a[0]*b, a[1]*b, a[2]*b]+[b] for a,b in zip(data,pesos)]

In [ ]:
surf = NURBS.Surface()

# Para los grados
surf.degree_u = 2
surf.degree_v = 1

# Puntos homogéneos: [x*w, y*w, z*w, w] donde w son los pesos
ctrlptsw = data_hom

# Asignar puntos con pesos directamente
surf.set_ctrlpts(ctrlptsw, m, n)  

# Vectores de nodos
surf.knotvector_u = x1
surf.knotvector_v = x2

# Densidad de puntos
surf.delta = 1/50

# Evaluación de puntos
surf.evaluate()
cinta_geomdl = surf.evalpts

At this stage, the points of the Möbius strip are stored in the variables **cinta_geomdl** and **cinta_nurbs**, where the former was computed using **geomdl** and the latter using the implemented class.

In the following lines, the root mean square error (**RMSE**) per point is calculated between these two lists of points.

In [ ]:
A = np.array(cinta_geomdl)
B = np.array(cinta_nurbs)

diff = A-B
sq_dist_per_point = np.sum(diff**2, axis=1)
mse_points = np.sqrt(np.mean(sq_dist_per_point))
print("RMSE por punto:", mse_points)

#### We visualize

In [ ]:
mobius = Nurbs_superficie3D_graph(data=my_data_cinta, pes1=w1, pes2=w2, nod1=x1, nod2=x2, p1=2, p2=1, ran_u=[0, 4], ran_v=[0, 1])
mobius.grafica(enmaller=True, pts_ctrl=False, superficie=True, cur=False)

## 5.2. Klein Bottle

For this example, the Klein bottle is considered with the following parametric equations:

$$ 
S(u,v) = 
\begin{bmatrix}
\left(a + \cos\left(\frac{u}{2}\right)\sin(v) - \sin\left(\frac{u}{2}\right)\sin(2v)\right)\cos(u) \\
\left(a + \cos\left(\frac{u}{2}\right)\sin(v) - \sin\left(\frac{u}{2}\right)\sin(2v)\right)\sin(u) \\
\sin\left(\frac{u}{2}\right)\sin(v) + \cos\left(\frac{u}{2}\right)\sin(2v)
\end{bmatrix},
\hspace{1mm} u \in [0,2\pi] \hspace{1mm} \wedge \hspace{1mm} v \in [0,4\pi]
$$

Below, the control points of the **direction** curve and the generatrix curve are specified. As mentioned earlier, the control points of the rotation and twisting curves are the same as those used for the Möbius strip.

In this case, the generatrix curve is a figure-eight curve (constructed with degree 4), and the **direction** curve is a circle of radius two, since $ a = 2 $.

In [ ]:
# ------------------------------------------------
# Definimos los puntos para la curva de direction
# ------------------------------------------------
direccion = Construccion(n=4, r=2).polygon()
direccion = [psdr + [0] for psdr in direccion]  


# -----------------------------------------------------
# Definimos los puntos para la curva generatriz (pts2) 
# -----------------------------------------------------

# Corresponde a la curva del ocho
gen= np.array([[0,0,0], [1/2,0,1], [3/4,0,3/2], [1,0,1], [1,0,0], [1,0,-1], [3/4,0,-3/2],
               [1/2,0,-1], [0,0,0], [-1/2,0,1], [-3/4,0,3/2], [-1,0,1], [-1,0,0], [-1,0,-1],
               [-3/4,0,-3/2], [-1/2,0,-1], [0,0,0]
    
])

# Pesos de la curva generatriz
w2 = [1,1,4/3,2,4,2,4/3,1,1,1,4/3,2,4,2,4/3,1,1]

# Nodos de la curva generatriz 
x2 = [0,0,0,0,0,1,1,1,1,2,2,2,2,3,3,3,3,4,4,4,4,4]

#### Data Generation:

The data (control mesh) for the Klein bottle is obtained by executing the following lines.

In [ ]:
ob = generation_data(pts1=rot, pts2=gen, pts3=gir, direction=direccion)
my_data_klein = ob.data

#### Evaluation of the NURBS Surface

Once the data has been generated, the NURBS surface can be evaluated using the class **Nurbs_surface**. First, it is initialized, and then it is evaluated with the method **procesar**.

Note that $ u \in [0,4] $ and $ v \in [0,4] $, since the corresponding knot vectors in both directions are:
$ [0,0,0,1,1,2,2,3,3,4,4,4] \wedge [0,0,0,0,0,1,1,1,1,2,2,2,2,3,3,3,3,4,4,4,4,4] $.

It is advisable to subtract $ 10^{-10} $ to avoid division by zero. The following lines evaluate the NURBS surface with a resolution of $ 50 \times 50 $.  

In this case, the degree of the generatrix curve is two.

In [ ]:
# Rango de parámetros u y v
u = np.linspace(0, 4-10**(-10), 50)
v = np.linspace(0, 4-10**(-10), 50)
U, V = np.meshgrid(u,v)

# Evaluación de superficie NURBS
botella = Nurbs_surface(my_data_klein, w1, w2, x1, x2, 2, 4)
X, Y, Z  = botella.procesar(U, V)

# Se cambia el formato a tamaño 2500x3 puntos
puntos1 = np.stack((X, Y, Z), axis=-1)
botella_nurbs = puntos1.reshape(-1, 3, order='F')

##### Verification

Next, the normal vectors along a selected trajectory of the surface are visualized.  

If the normals reverse their direction, this confirms that the data indeed corresponds to a non-orientable surface.

In [ ]:
normal_graph(X, Y, Z)

#### Validation

First, let us evaluate the Klein bottle using its parametric equations with a resolution of $ 500 \times 500 $.

In [ ]:
# Rango de parámetros u y v
u = np.linspace(0, 2*np.pi, 500)
v = np.linspace(0, 2*np.pi, 500)
U, V = np.meshgrid(u,v)

# Evaluamos la superficie paramétrica
a = 2
X = (a + np.cos(U/2)*np.sin(V) - np.sin(U/2)*np.sin(2*V))*np.cos(U)
Y = (a + np.cos(U/2)*np.sin(V) - np.sin(U/2)*np.sin(2*V))*np.sin(U)
Z = np.sin(U/2)*np.sin(V) + np.cos(U/2)*np.sin(2*V)

# Se cambia el formato a tamaño 250000x3 puntos
puntos2 = np.stack((X, Y, Z), axis=-1)
botella_parametrica = puntos2.reshape(-1, 3, order='F')

Next, we proceed to color the points of **botella_nurbs** according to the minimum distance to all points in **botella_parametrica**.

This allows us to measure the error between the data generated using NURBS and that obtained through parametric evaluation, and thus verify that the data **botella_nurbs** indeed corresponds to the Klein bottle.

In [ ]:
graficar_heatmap_error(botella_nurbs, botella_parametrica)

According to the plot, the error is concentrated in certain regions near the edges of the surface. This is expected, since the NURBS surface data consists of $ 50 \times 50 $ points, whereas the parametric evaluation data has a resolution of $ 500 \times 500 $. Moreover, this is a complex surface that self-intersects; a small amount of error appears precisely along the inner boundary where the surface bends.

Nevertheless, the vast majority of the points are displayed in dark colors, indicating that the error is nearly negligible.

#### Comparison with the geomdl Library

First, the data is homogenized; that is, the points that originally have three coordinates are converted into four coordinates by including the weight.

This is done in order to use **geomdl** and construct the NURBS surface from the homogeneous control point data.

In [ ]:
data = []
pesos = []
m = len(w1)
n = len(w2)

for i in range(m):
    for j in range(n):
        data.append(botella.P_ij(i,j))
        pesos.append(botella.W_ij(i,j))

data_hom = [[a[0]*b, a[1]*b, a[2]*b]+[b] for a,b in zip(data,pesos)]

In [ ]:
surf = NURBS.Surface()

# Para los grados
surf.degree_u = 2
surf.degree_v = 4

# Puntos homogéneos: [x*w, y*w, z*w, w] donde w son los pesos
ctrlptsw = data_hom

# Asignar puntos con pesos directamente
surf.set_ctrlpts(ctrlptsw, m, n)  

# Vectores de nodos
surf.knotvector_u = x1
surf.knotvector_v = x2

# Densidad de puntos
surf.delta = 1/50

# Evaluación de puntos
surf.evaluate()
botella_geomdl = surf.evalpts

At this stage, the points of the Klein bottle are stored in the variables **botella_geomdl** and **botella_nurbs**, where the former was computed using **geomdl** and the latter using the implemented class.

In the following lines, the root mean square error (**RMSE**) per point is calculated between these two lists of points.

In [ ]:
A = np.array(botella_geomdl)
B = np.array(botella_nurbs)

diff = A-B
sq_dist_per_point = np.sum(diff**2, axis=1)
mse_points = np.sqrt(np.mean(sq_dist_per_point))
print("RMSE por punto:", mse_points)

#### We visualize

In [ ]:
klein = Nurbs_superficie3D_graph(data=my_data_klein, pes1=w1, pes2=w2, nod1=x1, nod2=x2, p1=2, p2=4, ran_u=[0, 4], ran_v=[0, 4])
klein.grafica(enmaller=True, pts_ctrl=False, superficie=True, cur=False)

## 5.3. Approximate Parabolic Generatrix

The generatrix curve is an approximation of a parabola defined by three control points and a degree of two.

In [ ]:
# ------------------------------------------------
# Definimos los puntos para la curva de direction
# ------------------------------------------------
direccion = Construccion(n=4, r=2).polygon()
direccion = [psdr + [0] for psdr in direccion]  


# -----------------------------------------------------
# Definimos los puntos para la curva generatriz (pts2) 
# -----------------------------------------------------

gen= np.array([[-1,0,1/2], [0,0,-1/2], [1,0,1/2]])

w2 = [1,5,1]

x2 = [0,0,0,1,1,1]

#### Data Generation

In [ ]:
ob = generation_data(pts1=rot, pts2=gen, pts3=gir, direction=direccion)
my_data_surf1 = ob.data

#### Evaluation and verification

In [ ]:
# Rango de parámetros u y v
u = np.linspace(0, 4-10**(-10), 50)
v = np.linspace(0, 1-10**(-10), 50)
U, V = np.meshgrid(u,v)

# Evaluación de superficie NURBS
surf1 = Nurbs_surface(my_data_surf1, w1, w2, x1, x2, 2, 2)
X, Y, Z  = surf1.procesar(U, V)

# Se cambia el formato a tamaño 2500x3 puntos
puntos1 = np.stack((X, Y, Z), axis=-1)
surf1_nurbs = puntos1.reshape(-1, 3, order='F')

In [ ]:
normal_graph(X, Y, Z)

#### Visualization

In [ ]:
surf1 = Nurbs_superficie3D_graph(data=my_data_surf1, pes1=w1, pes2=w2, nod1=x1, nod2=x2, p1=2, p2=2, ran_u=[0, 4], ran_v=[0, 1])
surf1.grafica(enmaller=True, pts_ctrl=False, superficie=True, cur=False)

## 5.4. Heart-Shaped Generatrix

The degree of this generatrix curve has been defined using 9 control points and degree 3.

In [ ]:
# ------------------------------------------------
# Definimos los puntos para la curva de direction
# ------------------------------------------------
direccion = Construccion(n=4, r=2).polygon()
direccion = [psdr + [0] for psdr in direccion]  


# -----------------------------------------------------
# Definimos los puntos para la curva generatriz (pts2) 
# -----------------------------------------------------

gen= np.array([[0,0,-3/2], [-3/2,0,0], [-3/2,0,1], [-8/10,0,3/2], [0,0,1/2], 
               [8/10,0,3/2], [3/2,0,1], [3/2,0,0], [0,0,-3/2] ])

w2 = [1,1,1,1,1,1,1,1,1]

x2 = [0,0,0,0,1,2,3,4,5,6,6,6,6]

#### Data Generation

In [ ]:
ob = generation_data(pts1=rot, pts2=gen, pts3=gir, direction=direccion)
my_data_surf2 = ob.data

#### Evaluation and Verification

In [ ]:
# Rango de parámetros u y v
u = np.linspace(0, 4-10**(-10), 50)
v = np.linspace(0, 6-10**(-10), 50)
U, V = np.meshgrid(u,v)

# Evaluación de superficie NURBS
surf2 = Nurbs_surface(my_data_surf2, w1, w2, x1, x2, 2, 3)
X, Y, Z  = surf2.procesar(U, V)

# Se cambia el formato a tamaño 2500x3 puntos
puntos1 = np.stack((X, Y, Z), axis=-1)
surf2_nurbs = puntos1.reshape(-1, 3, order='F')

In [ ]:
normal_graph(X, Y, Z)

#### Visualization

In [ ]:
surf2 = Nurbs_superficie3D_graph(data=my_data_surf2, pes1=w1, pes2=w2, nod1=x1, nod2=x2, p1=2, p2=3, ran_u=[0, 3], ran_v=[0, 6])
surf2.grafica(enmaller=True, pts_ctrl=False, superficie=True, cur=False)

## 5.5.

The generatrix curve has 5 control points and degree 1.

In [ ]:
# ------------------------------------------------
# Definimos los puntos para la curva de direction
# ------------------------------------------------
direccion = Construccion(n=4, r=4).polygon()
direccion = [psdr + [0] for psdr in direccion]  


# -----------------------------------------------------
# Definimos los puntos para la curva generatriz (pts2) 
# -----------------------------------------------------

gen= np.array([[-2,0,0], [-1,0,3/2], [-3/2,0,0], [1,0,0], [1,0,2]])

w2 = [1,1,1,1,1]

x2 = [0,0,1,2,3,4,4]

#### Data Generation

In [ ]:
ob = generation_data(pts1=rot, pts2=gen, pts3=gir, direction=direccion)
my_data_surf3 = ob.data

#### Visualization

In [ ]:
surf3 = Nurbs_superficie3D_graph(data=my_data_surf3, pes1=w1, pes2=w2, nod1=x1, nod2=x2, p1=2, p2=1, ran_u=[0, 3], ran_v=[0, 4])
surf3.grafica(enmaller=True, pts_ctrl=False, superficie=True, cur=False)

## 5.6. Torus No Orientable
En comparación de un toro tradicional que se genera rotando una circunferencia alrededor de un eje, esta superficie también permite que la circunferencia gire al mismo tiempo que rota. Se considera a la curva generatriz como una circunferencia de radio $ a=1 $

In [ ]:
# ------------------------------------------------
# Definimos los puntos para la curva de direction
# ------------------------------------------------
direccion = Construccion(n=4, r=2).polygon()
direccion = [psdr + [0] for psdr in direccion] 


# -----------------------------------------------------
# Definimos los puntos para la curva generatriz (pts2) 
# -----------------------------------------------------

gen= Construccion(n=4, r=1).polygon() 
gen = [[psdr[0]]+[0]+[psdr[1]] for psdr in gen]

w2 = Construccion(n=4, r=1).peso()

x2 = Construccion(n=4, r=1).open1()

#### Data Generation

In [ ]:
ob = generation_data(pts1=rot, pts2=gen, pts3=gir, direction=direccion)
my_data_toro = ob.data

#### Evaluation and Verification

In [ ]:
# Rango de parámetros u y v
u = np.linspace(0, 4-10**(-10), 50)
v = np.linspace(0, 4-10**(-10), 50)
U, V = np.meshgrid(u,v)

# Evaluación de superficie NURBS
toro = Nurbs_surface(my_data_toro, w1, w2, x1, x2, 2, 2)
X, Y, Z  = toro.procesar(U, V)

# Se cambia el formato a tamaño 2500x3 puntos
puntos1 = np.stack((X, Y, Z), axis=-1)
toro_nurbs = puntos1.reshape(-1, 3, order='F')

In [ ]:
normal_graph(X, Y, Z)

#### Visualization

In [ ]:
toro = Nurbs_superficie3D_graph(data=my_data_toro, pes1=w1, pes2=w2, nod1=x1, nod2=x2, p1=2, p2=2, ran_u=[0, 4], ran_v=[0, 4])
toro.grafica(enmaller=True, pts_ctrl=False, superficie=True, cur=False)